# Capítulo 1 · Qubits y Estados Cuánticos

## Objetivos

Al finalizar este notebook el lector será capaz de:

1. Representar matemáticamente un qubit como vector en $\mathbb{C}^2$.
2. Aplicar puertas de un qubit (H, X, Y, Z, S, T) mediante multiplicación de matrices.
3. Calcular probabilidades de medida y el vector de Bloch.
4. Visualizar cualquier estado de un qubit en la esfera de Bloch.

---

## 1.1 El qubit como vector complejo

El estado general de un qubit se escribe como:

$$|\psi\rangle = \alpha|0\rangle + \beta|1\rangle, \quad \alpha,\beta \in \mathbb{C}, \quad |\alpha|^2 + |\beta|^2 = 1$$

donde los estados de la base computacional son:

$$|0\rangle = \begin{pmatrix}1\\0\end{pmatrix}, \quad |1\rangle = \begin{pmatrix}0\\1\end{pmatrix}$$

El estado $|\psi\rangle$ vive en el espacio de Hilbert $\mathcal{H} = \mathbb{C}^2$ y queda determinado (salvo fase global) por los ángulos esféricos $(\theta, \varphi)$:

$$|\psi\rangle = \cos\frac{\theta}{2}|0\rangle + e^{i\varphi}\sin\frac{\theta}{2}|1\rangle$$

Esta parametrización define la **esfera de Bloch**.

In [ ]:
# ── Dependencias ─────────────────────────────────────────────────
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..', '..'))  # acceso a src/

import numpy as np
import matplotlib.pyplot as plt
from src.quantum_math import QuantumMath
from src.quantum_gates import Gates
from src.visualization import QuantumVisualization

print('Módulos importados correctamente.')

## 1.2 Construcción de estados básicos

In [ ]:
# Estado base |0⟩
ket0 = QuantumMath.ket0()
# Estado base |1⟩
ket1 = QuantumMath.ket1()
# Superposición |+⟩ = (|0⟩ + |1⟩)/√2
ket_plus = QuantumMath.ket_plus()
# Superposición |-⟩ = (|0⟩ - |1⟩)/√2
ket_minus = QuantumMath.ket_minus()

print(f'|0⟩ = {ket0}')
print(f'|1⟩ = {ket1}')
print(f'|+⟩ = {ket_plus}')
print(f'|-⟩ = {ket_minus}')

# Verificación de normalización
for name, state in [('|0⟩',ket0),('|1⟩',ket1),('|+⟩',ket_plus),('|-⟩',ket_minus)]:
    norm = np.linalg.norm(state)
    print(f'  ||{name}|| = {norm:.6f}')

## 1.3 Puertas de un qubit

Una puerta cuántica de un qubit es una matriz **unitaria** $U \in \mathcal{U}(2)$. Las más importantes son:

| Puerta | Matriz | Efecto |
|--------|--------|--------|
| $X$ | $\begin{pmatrix}0&1\\1&0\end{pmatrix}$ | Inversión: $|0\rangle\leftrightarrow|1\rangle$ |
| $H$ | $\frac{1}{\sqrt{2}}\begin{pmatrix}1&1\\1&-1\end{pmatrix}$ | Superposición uniforme |
| $Z$ | $\begin{pmatrix}1&0\\0&-1\end{pmatrix}$ | Inversión de fase |
| $S$ | $\begin{pmatrix}1&0\\0&i\end{pmatrix}$ | Fase $\pi/2$ |
| $T$ | $\begin{pmatrix}1&0\\0&e^{i\pi/4}\end{pmatrix}$ | Fase $\pi/4$ |

In [ ]:
# Verificar unitariedad de cada puerta
for name, gate in [('H', Gates.H), ('X', Gates.X), ('Y', Gates.Y),
                   ('Z', Gates.Z), ('S', Gates.S), ('T', Gates.T)]:
    unitary = Gates.is_unitary(gate)
    print(f'Puerta {name}: unitaria = {unitary}')

print()
# Acción de H sobre |0⟩ → |+⟩
estado_inicial = QuantumMath.ket0()
tras_H = Gates.H @ estado_inicial
print(f'H|0⟩ = {np.round(tras_H, 4)}')

# Doble Hadamard recupera el estado original: H² = I
de_vuelta = Gates.H @ tras_H
print(f'H²|0⟩ = {np.round(de_vuelta, 4)}')

## 1.4 Probabilidades de medida

In [ ]:
# Estado arbitrario (definido por ángulos de Bloch)
theta_rad = np.radians(60)   # ángulo polar
phi_rad   = np.radians(45)   # ángulo azimutal

alpha = np.cos(theta_rad / 2)
beta  = np.exp(1j * phi_rad) * np.sin(theta_rad / 2)
psi   = np.array([alpha, beta])

print(f'Estado |ψ⟩ con θ=60°, φ=45°:')
print(f'  α = {alpha:.4f}')
print(f'  β = {beta.real:.4f} + {beta.imag:.4f}i')

probs = QuantumMath.probabilities(psi)
print(f'  P(|0⟩) = |α|² = {probs[0]:.4f}')
print(f'  P(|1⟩) = |β|² = {probs[1]:.4f}')
print(f'  Suma   = {np.sum(probs):.6f}')

# Medición simulada
counts = QuantumMath.measure(psi, n_shots=2048)
print(f'\nResultados de 2048 mediciones: {counts}')

In [ ]:
# Visualización del histograma de medidas
fig = QuantumVisualization.plot_histogram(
    counts,
    title=f'Distribución de medidas (θ=60°, φ=45°)',
)
plt.show()

## 1.5 Vector de Bloch y visualización 3D

In [ ]:
# Calcular el vector de Bloch
bv = QuantumMath.bloch_vector(psi)
print(f'Vector de Bloch para |ψ⟩:')
print(f'  x = {bv[0]:.4f}')
print(f'  y = {bv[1]:.4f}')
print(f'  z = {bv[2]:.4f}')
print(f'  ||r|| = {np.sqrt(sum(c**2 for c in bv)):.6f}  (debe ser 1 para estado puro)')

# Visualización estática en matplotlib
fig = QuantumVisualization.plot_bloch_vector(psi, title='Estado |ψ⟩ (θ=60°, φ=45°)')
plt.show()

## 1.6 Circuito equivalente en Qiskit

Ahora replicamos el mismo experimento con Qiskit para verificar la consistencia.

In [ ]:
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector
from qiskit_aer import AerSimulator

# Preparar el estado |ψ⟩ con Qiskit (usando initialize)
qc = QuantumCircuit(1)
qc.initialize([alpha, beta], 0)
qc.measure_all()

# Simulación
sim_backend = AerSimulator()
qc_transpiled = qc
job = sim_backend.run(qc, shots=2048)
result = job.result()
counts_qiskit = result.get_counts()

print('Conteos Qiskit:', counts_qiskit)

# Diagrama del circuito
qc_draw = QuantumCircuit(1)
qc_draw.initialize([alpha, beta], 0)
print(qc_draw.draw('text'))

## 1.7 Ejercicios propuestos

1. Construye el estado $|y+\rangle = \frac{|0\rangle + i|1\rangle}{\sqrt{2}}$ y calcula su vector de Bloch. ¿En qué eje de la esfera se sitúa?

2. Demuestra algebraicamente que $H X H = Z$. Verifícalo computacionalmente.

3. Aplica la secuencia $S \rightarrow T \rightarrow H$ sobre $|0\rangle$ y representa el estado resultante en la esfera de Bloch.

4. ¿Cuántas veces hay que aplicar la puerta $T$ para que $T^n = I$? Compruébalo numéricamente.

5. Calcula la fidelidad entre $|+\rangle$ y el estado obtenido tras aplicar $Rx(\pi/6)$ a $|0\rangle$.

In [ ]:
# Espacio de trabajo para los ejercicios
# ─────────────────────────────────────

# Ejercicio 2: HXH = Z
HXH = Gates.H @ Gates.X @ Gates.H
print('HXH =\n', np.round(HXH, 4))
print('Z   =\n', Gates.Z)
print('¿Son iguales?', np.allclose(HXH, Gates.Z))